# Computation B — GHZ–W interference

**It from Bit via Gödel, Paper 1** · companion to `app:qiskit-interference` · addresses **op:up-quark**, constrained by **op:associator-geodesic**

Prepare $|\psi\rangle = \alpha|\mathrm{GHZ}\rangle + \beta e^{i\varphi}|\mathrm{W}\rangle$ and
read the **Bloch radius $r$ of one cut** — the base coordinate the geodesic mass proxy is built
from. Two verified findings: *first-order response* ($r \approx \sqrt{2/3}\,\alpha\beta$ at
small admixture: a 2% W component gives a 17× enhancement over incoherent mixing) and *phase
rigidity* ($r$ independent of $\varphi$ to machine precision).

On hardware, $r$ is measured by **single-qubit tomography of the cut**: three expectation values
$\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ of $q_0$ via the Estimator, then
$r = \sqrt{\langle X\rangle^2 + \langle Y\rangle^2 + \langle Z\rangle^2}$. All $\beta^2$
points go to the device as multiple pubs in **one job**.

In [1]:
USE_HARDWARE = False   # flip to True to run on IBM hardware
SHOTS = 8192

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, partial_trace, SparsePauliOp

GHZ = np.zeros(8, complex); GHZ[0b000] = GHZ[0b111] = 1/np.sqrt(2)
W   = np.zeros(8, complex); W[0b001] = W[0b010] = W[0b100] = 1/np.sqrt(3)

def prep(b2, phi):
    a, b = np.sqrt(1 - b2), np.sqrt(b2)
    vec = a*GHZ + b*np.exp(1j*phi)*W
    qc = QuantumCircuit(3)
    qc.prepare_state(vec, [0, 1, 2])   # unitary state prep (ISA-friendly)
    return qc

B2S = [0.02, 0.05, 0.10, 0.50, 1.00]
OBS = [SparsePauliOp(p) for p in ("IIX", "IIY", "IIZ")]   # rightmost char = q0

## Exact baseline: the table and the phase-rigidity check

In [2]:
def bloch_r_exact(b2, phi):
    rho = partial_trace(Statevector(prep(b2, phi)), [1, 2])
    m = np.asarray(rho.data)
    return float(np.sqrt((2*m[0,1].real)**2 + (2*m[0,1].imag)**2 + (m[0,0]-m[1,1]).real**2))

print("beta^2   r (exact)   r (analytic)   r (incoherent)   enhancement")
for b2 in B2S:
    r  = bloch_r_exact(b2, 0.0)
    ra = np.sqrt((2/3)*(1-b2)*b2 + b2*b2/9)
    ri = b2/3
    print(f"{b2:5.2f}   {r:9.6f}   {ra:11.6f}   {ri:13.6f}   {r/ri:8.2f}x")

rs = [bloch_r_exact(0.10, p) for p in np.linspace(0, 2*np.pi, 25)]
print(f"\nphase rigidity at beta^2=0.10: spread over phi = {max(rs)-min(rs):.2e}")

beta^2   r (exact)   r (analytic)   r (incoherent)   enhancement
 0.02    0.114504      0.114504        0.006667      17.18x
 0.05    0.178730      0.178730        0.016667      10.72x
 0.10    0.247207      0.247207        0.033333       7.42x
 0.50    0.440959      0.440959        0.166667       2.65x
 1.00    0.333333      0.333333        0.333333       1.00x

phase rigidity at beta^2=0.10: spread over phi = 1.30e-15


## Sampled run — Estimator tomography, all points in one job

In [3]:
circs = [prep(b2, 0.0) for b2 in B2S]

if USE_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    service = QiskitRuntimeService()
    backend = service.least_busy(simulator=False, operational=True)
    print("backend:", backend.name)
    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    pubs = []
    for c in circs:
        isa = pm.run(c)
        pubs.append((isa, [o.apply_layout(isa.layout) for o in OBS]))
    result = Estimator(mode=backend).run(pubs).result()
else:
    from qiskit.primitives import StatevectorEstimator
    result = StatevectorEstimator().run([(c, OBS) for c in circs]).result()

print("beta^2   r (sampled)   r (analytic)")
for b2, res in zip(B2S, result):
    evs = np.asarray(res.data.evs, dtype=float)
    r = float(np.sqrt(np.sum(evs**2)))
    ra = np.sqrt((2/3)*(1-b2)*b2 + b2*b2/9)
    print(f"{b2:5.2f}   {r:11.6f}   {ra:11.6f}")

beta^2   r (sampled)   r (analytic)
 0.02      0.114504      0.114504
 0.05      0.178730      0.178730
 0.10      0.247207      0.247207
 0.50      0.440959      0.440959
 1.00      0.333333      0.333333


## Reading, and what is still missing

The cross term makes the mass-relevant invariant respond at **first order in the admixture
amplitude** — the right shape for a ~30% up-quark effect from a modest W component — and the
$\varphi$-independence means the mechanism, if right, lives in the admixture **magnitude**. What
this notebook does not supply is the map from $r$ to mass: that is exactly
**op:associator-geodesic**. The playground is built; the mechanism awaits its theorem.

Hardware notes: `prepare_state` on 3 qubits transpiles to a short two-qubit-gate sequence; the
dominant error on $r$ is readout bias on the three Paulis (≲ a few %). Sweep `phi` on hardware by
building `circs` with nonzero phase — rigidity should survive noise since it is a statement about
a magnitude.